# LLM Hidden-State & Logit Distillation Tutorial

This notebook demonstrates how to run the complete end-to-end knowledge distillation pipeline using the modular source code defined in the `src/` package. Both the teacher model (**Qwen2.5-3B-Instruct**) and student model (**Qwen2.5-0.5B-Instruct**) are loaded in quantized 4-bit config to fit easily on a standard T4 GPU.

### Setup & Installation
We start by installing the necessary package dependencies.

In [ ]:
# Install required libraries
!pip install -q torch transformers datasets accelerate peft scipy scikit-learn bitsandbytes matplotlib
!pip install -q --upgrade torchao

### 1. Import Packages
Import modules from `src/` along with PyTorch and utilities.

In [ ]:
import os
import gc
import json
import torch
import numpy as np
from transformers import AutoTokenizer, TrainingArguments, DataCollatorForSeq2Seq
from huggingface_hub import login

# Import modular components from src package
from src.config import HiddenDistillationConfig
from src.models import (
    load_teacher_model,
    load_student_model,
    HiddenStateProjection,
    merge_and_save_model,
    load_distilled_model
)
from src.dataset import InstructionDataset
from src.evaluator import DistillationEvaluator
from src.trainer import HiddenStateDistillationTrainer, TimeBudgetCallback
from src.utils import print_gpu_memory, plot_training_curves

### 2. Configure Distillation Settings
Load configurations and dynamically set batch sizes based on available GPU VRAM memory.

In [ ]:
config = HiddenDistillationConfig()

# Seed setting
torch.manual_seed(config.seed)
np.random.seed(config.seed)

# Determine GPU memory size and allocate batch sizes
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} | VRAM: {gpu_mem_gb:.2f} GB')
    device = 'cuda:0'
else:
    gpu_mem_gb = 0
    device = 'cpu'
    print('WARNING: Running on CPU!')

if gpu_mem_gb >= 35:
    per_device_train_batch_size, gradient_accumulation_steps = 8, 2
    config.gradient_checkpointing = False
elif gpu_mem_gb >= 20:
    per_device_train_batch_size, gradient_accumulation_steps = 4, 4
else:
    per_device_train_batch_size, gradient_accumulation_steps = 2, 8

print(f'Per Device Batch Size: {per_device_train_batch_size}')
print(f'Gradient Accumulation Steps: {gradient_accumulation_steps}')

### 3. Load Models & Projection Layer
Load the tokenizer, teacher, student models, and initialize the student hidden-state projection head.

In [ ]:
print_gpu_memory('Initial State')

# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.student_model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Load Models
teacher_model = load_teacher_model(config, device)
student_model = load_student_model(config, device)

# Load Projection Head
num_layers = 1 if config.match_layers == 'last' else student_model.config.num_hidden_layers
projection = HiddenStateProjection(
    teacher_hidden_size=teacher_model.config.hidden_size,
    student_hidden_size=student_model.config.hidden_size,
    projection_type=config.projection_type,
    num_layers=num_layers,
).to(device).to(torch.bfloat16)

print_gpu_memory('Models Loaded')

### 4. Run Baseline Evaluation
Measure validation perplexity, teacher-student KL divergence, and hidden similarity prior to training.

In [ ]:
val_dataset = InstructionDataset(
    config=config, tokenizer=tokenizer, max_length=config.max_seq_length,
    split='val', val_size=config.val_size, seed=config.seed
)

evaluator = DistillationEvaluator(
    teacher_model=teacher_model, student_model=student_model, projection=projection,
    tokenizer=tokenizer, config=config, device=device
)

baseline_metrics = evaluator.run_full_evaluation(val_dataset, epoch=-1)
print('✓ Baseline Evaluation Complete!')

### 5. Start Distillation Training
Load SFT instruction datasets, set parameters, and initiate training.

In [ ]:
train_dataset = InstructionDataset(
    config=config, tokenizer=tokenizer, max_length=config.max_seq_length,
    split='train', val_size=config.val_size, seed=config.seed
)
data_collator = DataCollatorForSeq2Seq(tokenizer, padding=True, max_length=config.max_seq_length, return_tensors='pt')

# HF Training Arguments
training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio,
    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    save_strategy=config.save_strategy,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    bf16=config.bf16,
    gradient_checkpointing=config.gradient_checkpointing,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to=config.report_to,
    eval_strategy='no',
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
)

trainer = HiddenStateDistillationTrainer(
    teacher_model=teacher_model,
    projection=projection,
    evaluator=evaluator,
    config=config,
    model=student_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    callbacks=[TimeBudgetCallback(config.time_budget_minutes)],
)

print('Starting training...')
trainer.train()
print('✓ Training Finished!')

### 6. Save Model and Final Evaluation
Export adapters, and run the final evaluation to display the summary comparison table.

In [ ]:
trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)

# Run post-training evaluation
final_metrics = evaluator.run_full_evaluation(val_dataset, epoch=config.num_train_epochs)

# Distillation Metrics Summary Table
print('\n' + '='*70)
print('DISTILLATION SUMMARY')
print('='*70)
print(f'\n{"Metric":<35} {"Before":<15} {"After":<15} {"Change":<15}')
print('-' * 70)
for name, key, lower_is_better in [
    ('Val Perplexity', 'val_perplexity', True),
    ('Teacher-Student KL', 'teacher_student_kl', True),
    ('Hidden Cosine Similarity', 'cosine_sim_avg', False),
]:
    before = baseline_metrics.get(key, float('nan'))
    after = final_metrics.get(key, float('nan'))
    change = after - before
    if not np.isnan(before) and not np.isnan(after):
        symbol = '✓' if (lower_is_better and change < 0) or (not lower_is_better and change > 0) else '✗'
        print(f'{name:<35} {before:<15.4f} {after:<15.4f} {change:+.4f} {symbol}')

### 7. Plot Curves & Merge Model Weights
Generate metric training plots, then merge student LoRA weights back into the student's base weights.

In [ ]:
# Plot validation curves
plot_training_curves(config.output_dir)

# Clean up memory to prepare for merge
del teacher_model
del student_model
del projection
del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Merge weights
merged_output_dir = './merged_model'
try:
    merge_and_save_model(config.student_model_name, config.output_dir, merged_output_dir)
except Exception as e:
    print(f'Error merging model: {e}')

### 8. Push to Hugging Face Hub (Optional)
Uncomment the cells below to upload your distilled model outputs to Hugging Face Hub.

In [ ]:
from huggingface_hub import notebook_login
# Run this cell to authenticate with Hugging Face interactively
# notebook_login()

In [ ]:
# Push LoRA Adapter
# lora_repo_id = "username/your-lora-repo"
# student_distilled, tokenizer_distilled = load_distilled_model(config.output_dir, config.student_model_name)
# student_distilled.push_to_hub(lora_repo_id)
# tokenizer_distilled.push_to_hub(lora_repo_id)

# Push Merged Model
# merged_repo_id = "username/your-merged-repo"
# from transformers import AutoModelForCausalLM
# merged_model = AutoModelForCausalLM.from_pretrained(merged_output_dir, trust_remote_code=True)
# merged_tokenizer = AutoTokenizer.from_pretrained(merged_output_dir, trust_remote_code=True)
# merged_model.push_to_hub(merged_repo_id)
# merged_tokenizer.push_to_hub(merged_repo_id)